In [ ]:
from datetime import datetime, timedelta, timezone 
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, DateType
import pyspark.sql.functions as F


schema = StructType([
    StructField("Date", DateType(), True),
    StructField("Meter", StringType(), True),
    StructField("ServiceName", StringType(), True),
    StructField("Cost", DoubleType(), True)
])

table_name = "prd.value_stream_monitoring.databricks_costs_daily"
staging_name = "prd.value_stream_monitoring.databricks_costs_staging"
delta_path = "abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_daily"
staging_path = "abfss://refinedzone@dls2adseus2edhprd01.dfs.core.windows.net/value_stream_monitoring/databricks_costs_staging"

try:
    df1 = spark.read.format("csv").option("header", "true").schema(schema).load(
        f"abfss://databricks@sasscaleservesavecusprd.dfs.core.windows.net/databricks_meter_cost_20260715.csv"
    )
except:
    df1 = None

spark.conf.set("spark.sql.shuffle.partitions", "10")

if df1 is None:
    df = None
    dbutils.notebook.exit("WARNING: No cost files found")
else:
    df = df1
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("path", staging_path) \
        .saveAsTable(staging_name)

    if not spark.catalog.tableExists(table_name):
        spark.sql(f"""
            CREATE TABLE {table_name}
            USING DELTA
            LOCATION '{delta_path}'
            AS SELECT * FROM {staging_name}
        """)
    else:
        spark.sql(f"""
            MERGE INTO {table_name} AS tgt
            USING {staging_name} AS src
            ON tgt.Date = src.Date
                AND tgt.`Meter` = src.`Meter`
                AND tgt.'ServiceName' = src.'ServiceName'
            WHEN MATCHED THEN UPDATE SET
                tgt.Cost = src.Cost
            WHEN NOT MATCHED THEN INSERT *
        """)

    spark.sql(f"DROP TABLE IF EXISTS {staging_name}")
else:
    print("No data to process")